<a href="https://colab.research.google.com/github/luismiguelaristi/MecanismosPythonUPB/blob/main/ManipuladorParallelo_2GDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis cinemático de Manipulador Paralelo Plano 2-GDL

![manipulador_paralelo.png](https://via.placeholder.com/400x300?text=Manipulador+Paralelo+Plano)

## Descripción
Un manipulador paralelo plano de 2 grados de libertad (GDL) es un mecanismo conformado por una base fija, dos cadenas cinemáticas idénticas, y un efector final que se mueve en el plano. Este tipo de manipuladores ofrecen ventajas como alta rigidez, carga útil elevada y velocidad de movimiento. A diferencia de los manipuladores seriales, los paralelos reparten la carga entre todos sus eslabones, lo que permite diseños más compactos y precisos.

### Estructura del Mecanismo (Configuración RRRRR)

El mecanismo analizado contiene **5 barras totales**: 1 chasis (barra fija) + 4 barras binarias

**Barras del mecanismo:**
- **B1**: Barra 1 - Chasis (marco base fijo)
- **B2**: Barra 2 - Brazo/Manivela izquierda (eslabón actuado)
- **B3**: Barra 3 - Eslabón acoplador izquierdo
- **B4**: Barra 4 - Brazo/Manivela derecha (eslabón actuado)
- **B5**: Barra 5 - Eslabón acoplador derecho

**Pares revolucionarios**: 5 en total (configuración RRRRR)
- **R1**: Chasis (B1) ↔ Barra 2 (en punto O₁ - base izquierda)
- **R2**: Barra 2 ↔ Barra 3 (en punto A₁)
- **R3**: Chasis (B1) ↔ Barra 4 (en punto O₂ - base derecha)
- **R4**: Barra 4 ↔ Barra 5 (en punto A₂)
- **R5**: Barra 3 ↔ Barra 5 (en punto P - efector final)

## Diagrama Cinemático

```
               P (R5: Efector Final)
              /|\
             / | \
           B3  |  B5     ← Eslabones acopladores
           /   |   \
          /    |    \
        A1     |    A2   ← Puntos de conexión
        / (R2) | (R4) \
       /       |       \
      B2      |        B4   ← Brazos actuadores
      /|      |       |×\
   (R1)|      |       |(R3)
 O1──────base────────O2     ← Chasis (B1)
  θ1              θ2
```

## Nomenclatura del Mecanismo

- **B1**: Barra 1 (Chasis) - fija
- **B2**: Barra 2 (Brazo izquierdo) - actuador 1
- **B3**: Barra 3 (Eslabón acoplador izquierdo)
- **B4**: Barra 4 (Brazo derecho) - actuador 2
- **B5**: Barra 5 (Eslabón acoplador derecho)
- **O₁**: Par revoluta en base izquierda
- **O₂**: Par revoluta en base derecha
- **A₁**: Punto de unión entre B2 y B3 (par revoluta R2)
- **A₂**: Punto de unión entre B4 y B5 (par revoluta R4)
- **P**: Punto de unión entre B3 y B5 (par revoluta R5) - Efector final

## Parámetros del Mecanismo

- $(x_{A2}, y_{A2})$: Posición del punto A₂

### Longitudes de Eslabones- $(x_{A1}, y_{A1})$: Posición del punto A₁

- $L_2$: Longitud del brazo izquierdo (B2)- $(x_P, y_P)$: Coordenadas cartesianas del efector final (P)

- $L_3$: Longitud del acoplador izquierdo (B3)### Variables de Salida (Posiciones)

- $L_4$: Longitud del brazo derecho (B4)

- $L_5$: Longitud del acoplador derecho (B5)- $\theta_2$: Ángulo de rotación del brazo derecho (B4)

- $d$: Distancia entre pares base O₁ y O₂- $\theta_1$: Ángulo de rotación del brazo izquierdo (B2)

### Variables de Entrada (Ángulos)

## Ecuaciones Vectoriales

### Cinemática Directa (Forward Kinematics)

La posición del efector final P se obtiene resolviendo el sistema de ecuaciones de restricción dados los ángulos de entrada $\theta_1$ y $\theta_2$.

**Posición del punto A₁ (unión B2-B3):**
$$A_1 = O_1 + L_2(\cos\theta_1, \sin\theta_1)$$
$$O_1 = (-d/2, 0)$$
$$A_{1x} = -d/2 + L_2\cos\theta_1$$
$$A_{1y} = L_2\sin\theta_1$$

**Posición del punto A₂ (unión B4-B5):**
$$A_2 = O_2 + L_4(\cos\theta_2, \sin\theta_2)$$
$$O_2 = (d/2, 0)$$
$$A_{2x} = d/2 + L_4\cos\theta_2$$
$$A_{2y} = L_4\sin\theta_2$$

**Efector final (punto P - intersección de B3 y B5):**

El punto P es la intersección de dos círculos:
- Círculo 1: Centro en $A_1$, radio $L_3$ (longitud de B3)
- Círculo 2: Centro en $A_2$, radio $L_5$ (longitud de B5)

$$|P - A_1| = L_3 \quad \text{(restricción de B3)}$$
$$|P - A_2| = L_5 \quad \text{(restricción de B5)}$$

Resolviendo este sistema de dos ecuaciones con dos incógnitas:
$$(x_P + d/2 - L_2\cos\theta_1)^2 + (y_P - L_2\sin\theta_1)^2 = L_3^2$$
$$(x_P - d/2 - L_4\cos\theta_2)^2 + (y_P - L_4\sin\theta_2)^2 = L_5^2$$

## Implementación Numérica

### Importar módulos

In [ ]:
%reset -sf

import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
import pandas as pd

### Parámetros del Mecanismo (Modificables)

Modifique los siguientes parámetros para cambiar las dimensiones del manipulador paralelo:

In [ ]:
# =====================================================
# PARÁMETROS MODIFICABLES DEL MECANISMO RRRRR
# =====================================================

# Longitudes de los eslabones [mm]
L2 = 100.0  # Longitud del brazo izquierdo (B2)
L3 = 80.0   # Longitud del acoplador izquierdo (B3)
L4 = 100.0  # Longitud del brazo derecho (B4)
L5 = 80.0   # Longitud del acoplador derecho (B5)

# Distancia entre pares revolucionarios en la base [mm]
d = 120.0   # Separación entre base O₁ (izquierda) y O₂ (derecha)

# Condiciones iniciales para análisis
theta1_ini = np.radians(45)   # Ángulo inicial del brazo izquierdo (θ₁) [rad]
theta2_ini = np.radians(135)  # Ángulo inicial del brazo derecho (θ₂) [rad]

# Información del mecanismo
print("="*60)
print("PARÁMETROS DEL MANIPULADOR PARALELO PLANO 2-GDL (RRRRR)")
print("="*60)
print("\n📊 BARRAS:")
print(f"  B1 (Chasis):                Fijo")
print(f"  B2 (Brazo izq.):            {L2} mm")
print(f"  B3 (Acoplador izq.):        {L3} mm")
print(f"  B4 (Brazo der.):            {L4} mm")
print(f"  B5 (Acoplador der.):        {L5} mm")
print(f"\n📍 GEOMETRÍA:")
print(f"  Separación de base (d):     {d} mm")
print(f"  Ubicación O₁ (base izq.):   (-{d/2}, 0)")
print(f"  Ubicación O₂ (base der.):   (+{d/2}, 0)")
print(f"\n⚙️  CONDICIONES INICIALES:")
print(f"  Ángulo θ₁ (brazo izq.):     {np.degrees(theta1_ini):.2f}°")
print(f"  Ángulo θ₂ (brazo der.):     {np.degrees(theta2_ini):.2f}°")
print("="*60)

### Cinemática Directa (Forward Kinematics)

In [ ]:
def cinemática_directa(theta1, theta2, x_p_ini=0, y_p_ini=100):
    """
    Calcula la posición del efector final (P) dado los ángulos de entrada.
    Resuelve el sistema de dos ecuaciones no lineales usando fsolve.
    
    Parámetros:
    -----------
    theta1 : float
        Ángulo de rotación del brazo izquierdo B2 (θ₁) [rad]
    theta2 : float
        Ángulo de rotación del brazo derecho B4 (θ₂) [rad]
    x_p_ini, y_p_ini : float
        Valores iniciales para el solucionador numérico
    
    Retorna:
    --------
    tuple : (x_p, y_p, A1_x, A1_y, A2_x, A2_y)
        Posición del efector final y puntos de acoplamiento
    """
    
    def ecuaciones_restriccion(vars, theta1, theta2):
        """
        Sistema de dos ecuaciones de restricción:
        - Distancia de P a A₁ debe ser L3
        - Distancia de P a A₂ debe ser L5
        """
        x_p, y_p = vars
        
        # Posiciones de A₁ (extremo de B2)
        A1_x = -d/2 + L2 * np.cos(theta1)
        A1_y = L2 * np.sin(theta1)
        
        # Posiciones de A₂ (extremo de B4)
        A2_x = d/2 + L4 * np.cos(theta2)
        A2_y = L4 * np.sin(theta2)
        
        # Ecuación 1: |P - A₁| = L3
        eq1 = (x_p - A1_x)**2 + (y_p - A1_y)**2 - L3**2
        
        # Ecuación 2: |P - A₂| = L5
        eq2 = (x_p - A2_x)**2 + (y_p - A2_y)**2 - L5**2
        
        return [eq1, eq2]
    
    # Resolver el sistema
    try:
        sol = fsolve(ecuaciones_restriccion, [x_p_ini, y_p_ini], args=(theta1, theta2), full_output=True)
        x_p, y_p = sol[0]
        info = sol[1]
        
        # Verificar convergencia
        if np.max(np.abs(info['fvec'])) > 1e-4:
            # Si no converge bien, intentar otra condición inicial
            sol = fsolve(ecuaciones_restriccion, [0, -100], args=(theta1, theta2), full_output=True)
            x_p, y_p = sol[0]
    except:
        x_p, y_p = x_p_ini, y_p_ini
    
    # Calcular posiciones de A₁ y A₂ para retornar
    A1_x = -d/2 + L2 * np.cos(theta1)
    A1_y = L2 * np.sin(theta1)
    
    A2_x = d/2 + L4 * np.cos(theta2)
    A2_y = L4 * np.sin(theta2)
    
    return x_p, y_p, A1_x, A1_y, A2_x, A2_y

# Prueba con las condiciones iniciales
x_p, y_p, A1_x, A1_y, A2_x, A2_y = cinemática_directa(theta1_ini, theta2_ini)

print(f"\n✓ Análisis de Posición con θ₁={np.degrees(theta1_ini):.1f}°, θ₂={np.degrees(theta2_ini):.1f}°")
print(f"\n📍 Efector Final (P):")
print(f"   x_p = {x_p:7.2f} mm")
print(f"   y_p = {y_p:7.2f} mm")
print(f"\n📍 Puntos de Acoplamiento:")
print(f"   A₁ = ({A1_x:7.2f}, {A1_y:7.2f}) mm   (unión B2-B3)")
print(f"   A₂ = ({A2_x:7.2f}, {A2_y:7.2f}) mm   (unión B4-B5)")
print(f"\n✓ Distancias (Verificación de Restricciones):")
dist_P_A1 = np.sqrt((x_p - A1_x)**2 + (y_p - A1_y)**2)
dist_P_A2 = np.sqrt((x_p - A2_x)**2 + (y_p - A2_y)**2)
print(f"   |P - A₁| = {dist_P_A1:.4f} mm  (debe ser ≈ {L3} mm)")
print(f"   |P - A₂| = {dist_P_A2:.4f} mm  (debe ser ≈ {L5} mm)")

### Análisis del Espacio de Trabajo (Workspace Analysis)

In [ ]:
def estimar_espacio_trabajo(rango_theta1, rango_theta2, num_puntos=40):
    """
    Estima el espacio de trabajo barriendo los ángulos de entrada.
    
    Parámetros:
    -----------
    rango_theta1 : tuple
        (min, max) ángulo para brazo izquierdo (θ₁) [rad]
    rango_theta2 : tuple
        (min, max) ángulo para brazo derecho (θ₂) [rad]
    num_puntos : int
        Número de puntos en cada dirección
    
    Retorna:
    --------
    dict : {'x': ..., 'y': ...} - Coordenadas del espacio de trabajo
    """
    
    theta1_range = np.linspace(rango_theta1[0], rango_theta1[1], num_puntos)
    theta2_range = np.linspace(rango_theta2[0], rango_theta2[1], num_puntos)
    
    workspace_x = []
    workspace_y = []
    x_prev, y_prev = 0, 100  # Condición inicial para el primer punto
    
    for theta1 in theta1_range:
        for theta2 in theta2_range:
            x_p, y_p, _, _, _, _ = cinemática_directa(theta1, theta2, x_prev, y_prev)
            workspace_x.append(x_p)
            workspace_y.append(y_p)
            x_prev, y_prev = x_p, y_p  # Usar posición anterior como hint
    
    return {'x': np.array(workspace_x), 'y': np.array(workspace_y)}

# Calcular el espacio de trabajo
print("\n🔄 Calculando espacio de trabajo...")
rango_theta1 = (np.radians(0), np.radians(180))
rango_theta2 = (np.radians(0), np.radians(180))

workspace = estimar_espacio_trabajo(rango_theta1, rango_theta2, num_puntos=35)

print(f"✓ Espacio de trabajo calculado: {len(workspace['x'])} puntos")
print(f"  Rango en X: [{workspace['x'].min():.2f}, {workspace['x'].max():.2f}] mm")
print(f"  Rango en Y: [{workspace['y'].min():.2f}, {workspace['y'].max():.2f}] mm")
print(f"  Área aproximada: {(workspace['x'].max()-workspace['x'].min())*(workspace['y'].max()-workspace['y'].min()):.0f} mm²")

### Visualización del Espacio de Trabajo

In [ ]:
# Visualizar espacio de trabajo
plt.figure(figsize=(10, 8))
plt.scatter(workspace['x'], workspace['y'], c='blue', s=20, alpha=0.6, label='Espacio de trabajo')

# Marcar posición inicial del efector final
x_ini, y_ini, _, _, _, _ = cinemática_directa(theta1_ini, theta2_ini)
plt.plot(x_ini, y_ini, 'r*', markersize=15, label='Posición inicial')

# Marcar las bases de los actuadores
plt.plot([-d/2, d/2], [0, 0], 'ks', markersize=10, label='Actuadores')

plt.xlabel('x [mm]', fontsize=12)
plt.ylabel('y [mm]', fontsize=12)
plt.title('Espacio de Trabajo del Manipulador Paralelo Plano 2-GDL', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.show()

### Consolidación de Datos con Pandas

In [ ]:
def generar_data_cinemática(rango_theta1, theta2_fijo=None, num_posiciones=30):
    """
    Genera datos cinemáticos para un rango de movimiento del primer actuador.
    
    Parámetros:
    -----------
    rango_theta1 : tuple
        (min, max) ángulo para brazo izquierdo (θ₁) [rad]
    theta2_fijo : float
        Ángulo fijo para brazo derecho. Si es None, θ₂ = θ₁
    num_posiciones : int
        Número de posiciones a analizar
    
    Retorna:
    --------
    pd.DataFrame : Datos cinemáticos con todas las variables
    """
    
    theta1_array = np.linspace(rango_theta1[0], rango_theta1[1], num_posiciones)
    
    if theta2_fijo is None:
        theta2_array = theta1_array  # Ambos actuadores con el mismo ángulo
    else:
        theta2_array = np.full(num_posiciones, theta2_fijo)
    
    data = {
        'theta1_deg': np.degrees(theta1_array),
        'theta2_deg': np.degrees(theta2_array),
        'x_p': [],
        'y_p': [],
        'A1_x': [],
        'A1_y': [],
        'A2_x': [],
        'A2_y': [],
        'dist_P_A1': [],
        'dist_P_A2': []
    }
    
    x_prev, y_prev = 0, 100  # Condición inicial
    
    for theta1, theta2 in zip(theta1_array, theta2_array):
        x_p, y_p, A1_x, A1_y, A2_x, A2_y = cinemática_directa(theta1, theta2, x_prev, y_prev)
        
        dist_A1 = np.sqrt((x_p - A1_x)**2 + (y_p - A1_y)**2)
        dist_A2 = np.sqrt((x_p - A2_x)**2 + (y_p - A2_y)**2)
        
        data['x_p'].append(x_p)
        data['y_p'].append(y_p)
        data['A1_x'].append(A1_x)
        data['A1_y'].append(A1_y)
        data['A2_x'].append(A2_x)
        data['A2_y'].append(A2_y)
        data['dist_P_A1'].append(dist_A1)
        data['dist_P_A2'].append(dist_A2)
        
        x_prev, y_prev = x_p, y_p
    
    return pd.DataFrame(data)

# Generar datos con θ₂ = 90° fijo
print("\n🔄 Generando datos cinemáticos...")
df = generar_data_cinemática((np.radians(0), np.radians(180)), theta2_fijo=np.radians(90), num_posiciones=25)

print("\n✓ Datos cinemáticos generados (primeras 8 filas):")
print(df.head(8).to_string())

### Gráficas de Análisis Cinemático

In [ ]:
# Crear figura con subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Gráfica 1: X vs Theta1
axes[0, 0].plot(df['theta1_deg'], df['x_p'], 'b-o', linewidth=2, markersize=4)
axes[0, 0].set_xlabel('Ángulo del brazo izquierdo θ₁ [°]', fontsize=10)
axes[0, 0].set_ylabel('Posición x_p [mm]', fontsize=10)
axes[0, 0].set_title('Posición X del Efector Final')
axes[0, 0].grid(True, alpha=0.3)

# Gráfica 2: Y vs Theta1
axes[0, 1].plot(df['theta1_deg'], df['y_p'], 'g-o', linewidth=2, markersize=4)
axes[0, 1].set_xlabel('Ángulo del brazo izquierdo θ₁ [°]', fontsize=10)
axes[0, 1].set_ylabel('Posición y_p [mm]', fontsize=10)
axes[0, 1].set_title('Posición Y del Efector Final')
axes[0, 1].grid(True, alpha=0.3)

# Gráfica 3: Distancia A1-A2 vs Theta1
axes[1, 0].plot(df['theta1_deg'], df['distancia_A1A2'], 'r-o', linewidth=2, markersize=4)
axes[1, 0].axhline(y=L3, color='k', linestyle='--', label=f'L₃ = {L3} mm')
axes[1, 0].set_xlabel('Ángulo del brazo izquierdo θ₁ [°]', fontsize=10)
axes[1, 0].set_ylabel('Distancia A1-A2 [mm]', fontsize=10)
axes[1, 0].set_title('Distancia entre Puntos de Acoplamiento')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Gráfica 4: Trayectoria del efector final
axes[1, 1].plot(df['x_p'], df['y_p'], 'b-o', linewidth=2, markersize=6)
axes[1, 1].plot(df['x_p'][0], df['y_p'][0], 'g*', markersize=15, label='Inicio')
axes[1, 1].plot(df['x_p'][-1], df['y_p'][-1], 'r*', markersize=15, label='Final')
axes[1, 1].set_xlabel('x_p [mm]', fontsize=10)
axes[1, 1].set_ylabel('y_p [mm]', fontsize=10)
axes[1, 1].set_title('Trayectoria del Efector Final')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axis('equal')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### Animación del Mecanismo

Función para visualizar el movimiento del manipulador paralelo

In [ ]:
def visualizar_configuraciones(theta1_range, num_configs=9):
    """
    Visualiza varias configuraciones del mecanismo.
    
    Parámetros:
    -----------
    theta1_range : tuple
        (min, max) ángulo para visualizar [rad]
    num_configs : int
        Número de configuraciones a mostrar
    """
    
    theta1_values = np.linspace(theta1_range[0], theta1_range[1], num_configs)
    theta2_fixed = np.radians(90)
    
    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    axes = axes.flatten()
    
    # Calcular espacios de trabajo para contexto
    ws = estimar_espacio_trabajo(theta1_range, (theta2_fixed-np.radians(20), theta2_fixed+np.radians(20)), num_puntos=30)
    
    for idx, theta1 in enumerate(theta1_values):
        ax = axes[idx]
        
        # Dibujar espacio de trabajo (contexto)
        ax.scatter(ws['x'], ws['y'], c='lightblue', s=10, alpha=0.5, zorder=1)
        
        # Calcular posiciones
        x_p, y_p, A1_x, A1_y, A2_x, A2_y = cinemática_directa(theta1, theta2_fixed)
        
        # Dibujar eslabones
        # Brazo izquierdo
        ax.plot([-d/2, A1_x], [0, A1_y], 'b-', linewidth=3, label='Brazo izq.')
        # Brazo derecho
        ax.plot([d/2, A2_x], [0, A2_y], 'r-', linewidth=3, label='Brazo der.')
        # Acoplador
        ax.plot([A1_x, A2_x], [A1_y, A2_y], 'g-', linewidth=3, label='Acoplador')
        
        # Dibujar pares revolucionarios
        ax.plot([-d/2, d/2], [0, 0], 'ks', markersize=8, label='Base')
        ax.plot([A1_x, A2_x], [A1_y, A2_y], 'go', markersize=8, zorder=3)
        ax.plot(x_p, y_p, 'r*', markersize=15, label='Efector final', zorder=4)
        
        # Configurar ejes
        ax.set_xlim(-150, 150)
        ax.set_ylim(-50, 200)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('x [mm]', fontsize=9)
        ax.set_ylabel('y [mm]', fontsize=9)
        ax.set_title(f'θ₁ = {np.degrees(theta1):.1f}°, θ₂ = {np.degrees(theta2_fixed):.1f}°', fontsize=10)
        
        if idx == 0:
            ax.legend(fontsize=8, loc='upper left')
    
    # Ocultar ejes sobrantes
    for idx in range(len(theta1_values), 9):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualizar diferentes configuraciones
visualizar_configuraciones((np.radians(0), np.radians(180)), num_configs=9)

## Análisis de Velocidades

### Matriz Jacobiana

La relación entre velocidades articulares y velocidades cartesianas está dada por:

$$\begin{pmatrix} \dot{x} \\ \dot{y} \end{pmatrix} = \mathbf{J} \begin{pmatrix} \dot{\theta}_1 \\ \dot{\theta}_2 \end{pmatrix}$$

donde la matriz Jacobiana es:

$$\mathbf{J} = \begin{pmatrix} \frac{\partial x_p}{\partial \theta_1} & \frac{\partial x_p}{\partial \theta_2} \\ \frac{\partial y_p}{\partial \theta_1} & \frac{\partial y_p}{\partial \theta_2} \end{pmatrix}$$

In [ ]:
def calcular_jacobiana(theta1, theta2, delta=1e-6):
    """
    Calcula la matriz Jacobiana numéricamente.
    
    Parámetros:
    -----------
    theta1, theta2 : float
        Ángulos articulares [rad]
    delta : float
        Perturbación para derivada numérica
    
    Retorna:
    --------
    np.ndarray : Matriz Jacobiana 2x2
    """
    
    # Posición actual
    x0, y0, _, _, _, _ = cinemática_directa(theta1, theta2)
    
    # Derivadas respecto a theta1
    x1, y1, _, _, _, _ = cinemática_directa(theta1 + delta, theta2)
    dx_dth1 = (x1 - x0) / delta
    dy_dth1 = (y1 - y0) / delta
    
    # Derivadas respecto a theta2
    x2, y2, _, _, _, _ = cinemática_directa(theta1, theta2 + delta)
    dx_dth2 = (x2 - x0) / delta
    dy_dth2 = (y2 - y0) / delta
    
    J = np.array([[dx_dth1, dx_dth2],
                 [dy_dth1, dy_dth2]])
    
    return J

def calcular_dexteridad(theta1, theta2):
    """
    Calcula el índice de dexteridad (número de condición de la Jacobiana).
    
    Parámetros:
    -----------
    theta1, theta2 : float
        Ángulos articulares [rad]
    
    Retorna:
    --------
    float : Número de condición (condicionamiento de la matriz)
    """
    
    J = calcular_jacobiana(theta1, theta2)
    
    # Número de condición
    cond_number = np.linalg.cond(J)
    
    # Determinante
    det_J = np.linalg.det(J)
    
    return cond_number, det_J, J

# Calcular jacobiana en varias posiciones
print("\nAnálisis de Velocidades (Matriz Jacobiana)")
print("="*60)

test_angles = [
    (np.radians(45), np.radians(90)),
    (np.radians(90), np.radians(90)),
    (np.radians(135), np.radians(90))
]

for theta1, theta2 in test_angles:
    cond, det, J = calcular_dexteridad(theta1, theta2)
    print(f"\nθ₁ = {np.degrees(theta1):6.1f}°, θ₂ = {np.degrees(theta2):6.1f}°")
    print(f"  Matriz Jacobiana:\n{J}")
    print(f"  Determinante: {det:.4f}")
    print(f"  Número de Condición: {cond:.4f}")
    if abs(det) < 1e-3:
        print(f"  ⚠ SINGULARIDAD CERCANA")

### Mapa de Singularidades en el Espacio de Trabajo

In [ ]:
def mapear_dexteridad(rango_theta1, rango_theta2, num_puntos=30):
    """
    Crea un mapa de dexteridad del espacio de trabajo.
    """
    
    theta1_range = np.linspace(rango_theta1[0], rango_theta1[1], num_puntos)
    theta2_range = np.linspace(rango_theta2[0], rango_theta2[1], num_puntos)
    
    det_map = np.zeros((num_puntos, num_puntos))
    x_map = np.zeros((num_puntos, num_puntos))
    y_map = np.zeros((num_puntos, num_puntos))
    
    for i, theta1 in enumerate(theta1_range):
        for j, theta2 in enumerate(theta2_range):
            cond, det, _ = calcular_dexteridad(theta1, theta2)
            det_map[j, i] = det
            x_p, y_p, _, _, _, _ = cinemática_directa(theta1, theta2)
            x_map[j, i] = x_p
            y_map[j, i] = y_p
    
    return x_map, y_map, det_map, theta1_range, theta2_range

# Calcular mapas
x_map, y_map, det_map, th1_range, th2_range = mapear_dexteridad(
    (np.radians(0), np.radians(180)), 
    (np.radians(0), np.radians(180)), 
    num_puntos=25
)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfica 1: Determinante en espacio articular
im1 = axes[0].contourf(np.degrees(th1_range), np.degrees(th2_range), det_map, levels=20, cmap='RdBu_r')
contour = axes[0].contour(np.degrees(th1_range), np.degrees(th2_range), det_map, levels=[0], colors='black', linewidths=2)
axes[0].clabel(contour, inline=True, fontsize=10)
axes[0].set_xlabel('θ₁ [°]', fontsize=11)
axes[0].set_ylabel('θ₂ [°]', fontsize=11)
axes[0].set_title('Determinante Jacobiana (Espacio Articular)')
cbar1 = plt.colorbar(im1, ax=axes[0])
cbar1.set_label('det(J)', fontsize=10)

# Gráfica 2: Determinante en espacio cartesiano
im2 = axes[1].scatter(x_map, y_map, c=det_map, s=50, cmap='RdBu_r', alpha=0.7)
axes[1].set_xlabel('x [mm]', fontsize=11)
axes[1].set_ylabel('y [mm]', fontsize=11)
axes[1].set_title('Determinante Jacobiana (Espacio Cartesiano)')
axes[1].set_aspect('equal')
cbar2 = plt.colorbar(im2, ax=axes[1])
cbar2.set_label('det(J)', fontsize=10)

plt.tight_layout()
plt.show()

print("\nMapa de dexteridad completado.")

## Conclusiones y Observaciones

Este cuaderno proporciona un análisis completo de la cinemática de un manipulador paralelo plano de 2 grados de libertad. Los temas tratados incluyen:

1. **Cinemática Directa**: Cálculo de la posición del efector final dada los ángulos articulares.
2. **Espacio de Trabajo**: Visualización completa de las posiciones alcanzables.
3. **Análisis de Posición**: Gráficas que muestran la relación entre ángulos y posiciones.
4. **Configuraciones Múltiples**: Visualización de diferentes configuraciones del mecanismo.
5. **Análisis de Velocidades**: Cálculo de la matriz Jacobiana y análisis de singularidades.
6. **Mapas de Dexteridad**: Identificación de regiones singulares en el espacio de trabajo.

## Modificaciones Sugeridas

Los parámetros pueden modificarse en la celda de "Parámetros del Mecanismo" para explorar diferentes diseños:
- Cambiar longitudes de eslabones (L1, L2, L3)
- Ajustar la separación de actuadores (d)
- Modificar las condiciones iniciales de ángulos

## Referencias

- Craig, J. J. (2005). Introduction to Robotics: Mechanics and Control
- Tsai, L. W. (1999). Robot Analysis: The Mechanics of Serial and Parallel Manipulators
- Murray, R. M., Sastry, S. S., & Zexiang, L. (1994). A Mathematical Introduction to Robotic Manipulation